# Module 2: Accuracy and Predictive-Quality Improvement

This notebook optimizes the model using class weighting, oversampling, and hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/student_dropout_8_features.csv')
X = df.drop(columns=['dropout'])
y = df['dropout']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

continuous_cols = [
    'age_at_enrollment', 'admission_grade_pct', 
    'previous_semester_pass_ratio_pct', 'previous_semester_avg_grade_pct'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_cols)
    ],
    remainder='passthrough'
)

## 2. Hyperparameter Tuning with SMOTENC and Class Weighting

In [3]:
# Categorical indices for SMOTENC (gender, scholarship_holder, debtor, displaced)
# X columns order: 'age_at_enrollment', 'gender', 'admission_grade_pct', 'scholarship_holder', 'debtor', 'displaced', 'previous_semester_pass_ratio_pct', 'previous_semester_avg_grade_pct'
cat_indices = [1, 3, 4, 5]

smote = SMOTENC(categorical_features=cat_indices, random_state=42)

rf_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', smote),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

Best parameters: {'classifier__class_weight': None, 'classifier__max_depth': 10, 'classifier__n_estimators': 100}


## 3. Evaluation and Save Artifacts

In [4]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_prob)
}
print("Optimized Model Metrics:", json.dumps(metrics, indent=2))

pd.DataFrame([metrics]).to_csv('../reports/module2_results.csv', index=False)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Dropout', 'Dropout'], yticklabels=['Non-Dropout', 'Dropout'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - Optimized Model')
plt.tight_layout()
plt.savefig('../reports/confusion_matrix.png')
plt.close()

# Save final model
joblib.dump(best_model, '../artifacts/model.joblib')
print("Model saved to artifacts/model.joblib")

Optimized Model Metrics: {
  "accuracy": 0.8327683615819209,
  "precision": 0.7361111111111112,
  "recall": 0.7464788732394366,
  "f1": 0.7412587412587412,
  "roc_auc": 0.886322092287502
}


Model saved to artifacts/model.joblib
